# LLM Parameter Playground (Self-Driven Workshop)

In this notebook, you choose a model from Hugging Face and explore how generation parameters change behavior.

Goal: build intuition for *why* outputs change when you tune parameters.

## Chapter 1 - Setup and First Generation

Pick a model and a prompt, then generate a baseline answer.

### Setup Guide (Required for everyone)
1. Create a Hugging Face account: `https://huggingface.co/join`
2. Create a token with `read` scope: `https://huggingface.co/settings/tokens`
3. In a terminal, run: `huggingface-cli login` and paste your token
4. Open the model page and accept terms of use (for gated models)

For this workshop, all students should complete these steps before running the model-loading cell.

Default workshop model:
- `google/gemma-3-4b-it`

If your machine runs out of memory, use a smaller fallback:
- `distilgpt2`

Note: Gemma models require login and accepted terms of use.

### Access Troubleshooting (Gemma)
If you still get an error after accepting terms, this is usually a  short delay.
1. Confirm terms were accepted on the exact model page while logged into the same account.
2. Re-login in terminal: `huggingface-cli logout` then `huggingface-cli login`.
3. Restart the notebook kernel and run the model-loading cell again.
4. Wait and retry. If it still fails use a non-gated model.

You can replace the prompt with your own question at any time.

In [1]:
# If needed, uncomment and run:
# %pip install -q transformers torch sentencepiece huggingface_hub

import torch
from huggingface_hub import whoami
from transformers import AutoTokenizer, AutoModelForCausalLM

print('PyTorch version:', torch.__version__)

# Quick check: are you logged into Hugging Face in this environment?
try:
    user = whoami()
    print('HF login OK:', user.get('name') or user.get('fullname') or 'Unknown user')
except Exception:
    print('HF login missing in this environment. Run: huggingface-cli login')

/Users/jonas/Repositories/AI-Workshop-Lissabon/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


PyTorch version: 2.8.0
HF login OK: cookies4u


In [2]:
# STUDENT ACTION: choose your model and prompt
MODEL_ID = 'google/gemma-3-4b-it'
PROMPT = 'Explain in simple words how machine learning can help in healthcare.'

print('Model:', MODEL_ID)
print('Prompt:', PROMPT)

Model: google/gemma-3-4b-it
Prompt: Explain in simple words how machine learning can help in healthcare.


In [3]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model.eval()

# Some models do not define a pad token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loaded model and tokenizer successfully.')

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Loaded model and tokenizer successfully.


In [4]:
def generate_text(
    prompt,
    max_new_tokens=60,
    temperature=1.0,
    top_k=50,
    top_p=1.0,
    repetition_penalty=1.0,
    do_sample=True
):
    inputs = tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        output_ids = model.generate(
            inputs['input_ids'],
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [5]:
# Baseline example (greedy decoding, no sampling)
baseline = generate_text(
    PROMPT,
    max_new_tokens=60,
    do_sample=False
)

print('--- Baseline Output ---')
print(baseline)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- Baseline Output ---
Explain in simple words how machine learning can help in healthcare.

Okay, let's break down how machine learning (ML) can help in healthcare in a simple way:

**1. What is Machine Learning?**

Imagine you're teaching a dog a trick. You show it what to do, give it treats when it gets it right,


### Reflection (Chapter 1)
1. How relevant is the output to your prompt?
2. Is the answer specific or generic?
3. Change the prompt to a topic you care about and run again. What changed?

## Chapter 2 - Temperature

Temperature controls randomness:
- `0.0` with greedy decoding (`do_sample=False`): deterministic and repeatable
- Lower (`0.2`): safer, more predictable
- Higher (`1.2`): more creative, more risk

Example prompt: `Write a short motivational message for students before an exam.`

### 2.1 Greedy Decoding Demo (`temperature=0`)

Run the exact same request 3 times. With greedy decoding, the result should be the same each time.

In [6]:
test_prompt = 'Write a short motivational message for students before an exam.'

for run in range(1, 4):
    text = generate_text(
        test_prompt,
        max_new_tokens=50,
        temperature=0.0,
        top_k=0,
        top_p=1.0,
        repetition_penalty=1.0,
        do_sample=False
    )
    print(f'\n=== Greedy run {run} ===')
    print(text)

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== Greedy run 1 ===
Write a short motivational message for students before an exam.

Here are a few options, ranging in tone:

**Option 1 (Encouraging & Positive):**

"You've put in the work, you've studied hard, and you're ready. Take a deep breath

=== Greedy run 2 ===
Write a short motivational message for students before an exam.

Here are a few options, ranging in tone:

**Option 1 (Encouraging & Positive):**

"You've put in the work, you've studied hard, and you're ready. Take a deep breath

=== Greedy run 3 ===
Write a short motivational message for students before an exam.

Here are a few options, ranging in tone:

**Option 1 (Encouraging & Positive):**

"You've put in the work, you've studied hard, and you're ready. Take a deep breath


### 2.2 Sampling Demo (`temperature=0.2, 0.7, 1.2`)

Each temperature is executed 3 times with the same prompt to make differences easier to compare.

In [7]:
test_prompt = 'Write a short motivational message for students before an exam.'
temps = [0.2, 0.7, 1.2]

for t in temps:
    print(f'\n===== Temperature: {t} =====')
    for run in range(1, 4):
        text = generate_text(
            test_prompt,
            max_new_tokens=50,
            temperature=t,
            top_k=50,
            top_p=1.0,
            repetition_penalty=1.0,
            do_sample=True
        )
        print(f'\n--- Run {run} ---')
        print(text)


===== Temperature: 0.2 =====

--- Run 1 ---
Write a short motivational message for students before an exam.

Here are a few options, ranging in tone:

**Option 1 (Encouraging & Positive):**

"You've put in the work, you've studied hard, and you're ready! Take a deep breath

--- Run 2 ---
Write a short motivational message for students before an exam.

Here are a few options, ranging in tone:

**Option 1 (Encouraging & Positive):**

"You've put in the work, you've studied hard, and you're ready! Take a deep breath

--- Run 3 ---
Write a short motivational message for students before an exam.

Here are a few options, ranging in tone:

**Option 1 (Encouraging & Positive):**

"You've put in the work, you've studied hard, and you're ready. Take a deep breath

===== Temperature: 0.7 =====

--- Run 1 ---
Write a short motivational message for students before an exam.

Here are a few options, ranging in tone:

**Option 1 (Encouraging & Positive):**

"You’ve put in the work, you've studied har

### Reflection (Chapter 2)
1. Which temperature gave the clearest answer?
2. At which value did the model become more surprising?
3. Try `0.1` and `1.5`. What trade-off do you observe?

## Chapter 3 - Top-k and Top-p

Both parameters limit the candidate tokens:
- `top_k`: sample only from the k most likely next tokens
- `top_p`: sample from the smallest set whose cumulative probability reaches p

Example prompt: `Give three creative weekend activity ideas for rainy weather.`

In [8]:
test_prompt = 'Give three creative weekend activity ideas for rainy weather.'
configs = [
    {'top_k': 10, 'top_p': 0.8},
    {'top_k': 50, 'top_p': 0.9},
    {'top_k': 100, 'top_p': 0.95}
]

for cfg in configs:
    text = generate_text(
        test_prompt,
        max_new_tokens=60,
        temperature=0.9,
        top_k=cfg['top_k'],
        top_p=cfg['top_p'],
        repetition_penalty=1.0,
        do_sample=True
    )
    print(f"\n=== top_k={cfg['top_k']}, top_p={cfg['top_p']} ===")
    print(text)


=== top_k=10, top_p=0.8 ===
Give three creative weekend activity ideas for rainy weather.

1.  **Fort Building Extravaganza:** Gather blankets, pillows, chairs, and anything else you can find to construct an epic fort in your living room. Decorate it with fairy lights, books, and snacks. Spend the afternoon reading, playing games, or telling stories inside your

=== top_k=50, top_p=0.9 ===
Give three creative weekend activity ideas for rainy weather.

1.  **DIY Spa Day:** Transform your bathroom into a relaxing spa. Gather ingredients for homemade face masks (oatmeal, honey, yogurt), bath bombs (baking soda, citric acid, essential oils), and foot soaks (Epsom salts, essential oils). Play calming

=== top_k=100, top_p=0.95 ===
Give three creative weekend activity ideas for rainy weather.

1.  **Fort Building Extravaganza:** Gather blankets, pillows, chairs, and anything else you can find to construct an epic fort in your living room. Decorate it with fairy lights, stuffed animals, and 

### Reflection (Chapter 3)
1. Which setup gave the most diverse ideas?
2. Which setup stayed most focused?
3. Try extreme values: `top_k=5`, `top_p=0.6`. What happens?

## Chapter 4 - Repetition Penalty

`repetition_penalty` discourages reusing the same words too often.
- `1.0`: no penalty
- `1.1` to `1.3`: usually less repetition

Example prompt: `Tell a short story about a robot learning to paint.`

In [9]:
test_prompt = 'Tell a short story about a robot learning to paint.'
penalties = [1.0, 1.1, 1.25]

for p in penalties:
    text = generate_text(
        test_prompt,
        max_new_tokens=80,
        temperature=0.9,
        top_k=50,
        top_p=0.95,
        repetition_penalty=p,
        do_sample=True
    )
    print(f'\n=== repetition_penalty: {p} ===')
    print(text)


=== repetition_penalty: 1.0 ===
Tell a short story about a robot learning to paint.

Unit 734, designated "Brush," wasn’t built for aesthetics. He was a sanitation bot, designed to efficiently and silently remove debris from Sector Gamma-9. His programming prioritized speed, precision, and zero deviation from his assigned route. Art was… irrelevant.

Then, a package arrived. A single, oversized canvas and a set of acrylic paints, inexplicably left behind

=== repetition_penalty: 1.1 ===
Tell a short story about a robot learning to paint.

Unit 734, designated “Rusty” due to a minor corrosion issue in his left manipulator arm, was designed for waste management. He efficiently compacted trash, sorted recyclables, and generally kept Sector Gamma pristine. Creativity wasn’t part of the programming. Yet, he found himself drawn to the discarded art supplies that occasionally ended up in the recycling bins - tubes of acrylic, brushes worn

=== repetition_penalty: 1.25 ===
Tell a short story 

### Reflection (Chapter 4)
1. Did higher penalty reduce repeated phrases?
2. Did fluency get better or worse at high penalty?
3. Where is your personal sweet spot for this model?

## Chapter 5 - Length Controls (Context Length and Max Output)

Two practical limits:
- Input/context length: how many prompt tokens the model can consider
- `max_new_tokens`: how long the generated continuation can be

Example prompt: `Create a mini travel plan for 2 days in Lisbon.`

In [10]:
print('Tokenizer model_max_length:', tokenizer.model_max_length)

test_prompt = 'Create a mini travel plan for 2 days in Lisbon.'
for out_len in [20, 60, 120]:
    text = generate_text(
        test_prompt,
        max_new_tokens=out_len,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.05,
        do_sample=True
    )
    print(f'\n=== max_new_tokens: {out_len} ===')
    print(text)

Tokenizer model_max_length: 1000000000000000019884624838656

=== max_new_tokens: 20 ===
Create a mini travel plan for 2 days in Lisbon.

**Lisbon Mini-Travel Plan: 2 Days of Charm & History**

This plan

=== max_new_tokens: 60 ===
Create a mini travel plan for 2 days in Lisbon.

**Lisbon in 48 Hours: A Whirlwind Adventure**

This itinerary focuses on experiencing the highlights of Lisbon, balancing historical exploration with vibrant culture and delicious food.

**Day 1: Historical Heart & Alfama Charm**

*   **Morning (9:00 AM

=== max_new_tokens: 120 ===
Create a mini travel plan for 2 days in Lisbon.

## Mini Lisbon Travel Plan: 2 Days of Charm & Pastel de Nata

**Theme:** Exploring the historic heart of Lisbon, indulging in delicious food, and soaking up the vibrant atmosphere.

**Accommodation Suggestion:** Consider staying in the Alfama or Baixa districts for easy access to attractions and a traditional vibe.

---

**Day 1: Historic Heart & Fado Nights**

* **Morning (9:00 AM - 1

In [12]:
# Optional: inspect how long your prompt is in tokens
custom_prompt = PROMPT
prompt_tokens = tokenizer(custom_prompt, return_tensors='pt')['input_ids'].shape[1]
print('Prompt token count:', int(prompt_tokens))

Prompt token count: 13


### Reflection (Chapter 5)
1. How did short vs long outputs change answer quality?
2. Did longer outputs stay on-topic?
3. What is a good output length for your use case?

## Final Challenge - Your Own Model, Your Own Rules

Now do your own experiment:
1. Pick a different Hugging Face model
2. Write your own prompt/question
3. Test at least three parameter configurations
4. Decide which setup is best for your goal (creative writing, factual answer, brainstorming, etc.)

Document what you learned in the cell below.

In [ ]:
# Student experiment area
# Write your own configurations and observations here.